# Cài đặt thư viện


In [ ]:
!pip install transformers==4.37.2 accelerate==0.27.2 peft==0.8.2 datasets evaluate sentencepiece -q

import pandas as pd
import torch
import numpy as np
from datasets import Dataset
from transformers import T5ForConditionalGeneration, T5Tokenizer
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang chạy trên thiết bị: {device}")

In [ ]:
import json
import pandas as pd
from datasets import Dataset

file_path = "/kaggle/input/datasets/balancelott/acos-dataset/full_data_10k.jsonl" 

processed_data = []

# Đọc file jsonl từng dòng
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
            
        item = json.loads(line.strip())
        review_text = item.get("review_text", "")
        
        if "annotations" in item and item["annotations"]:
            acos_list = []
            
            for ann in item["annotations"]:

                acos_item = {
                    "aspect_expression": ann.get("aspect_expression", "NULL"),
                    "aspect_category": ann.get("aspect_category", "NULL"),
                    "sentiment": ann.get("sentiment", "NULL")
                }
                acos_list.append(acos_item)
            
            if acos_list:
                
                input_text = f"review: {review_text}"
                

                target_text = json.dumps(acos_list, ensure_ascii=False)
                
                processed_data.append({
                    "input_text": input_text,
                    "target_text": target_text
                })

df = pd.DataFrame(processed_data)
dataset = Dataset.from_pandas(df)

split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Số lượng Train: {len(train_dataset)}, Số lượng Eval: {len(eval_dataset)}")
print("\n[Ví dụ Input]:", train_dataset[0]["input_text"])
print("[Ví dụ Target]:", train_dataset[0]["target_text"])

In [ ]:
from transformers import AutoTokenizer

teacher_model_name = "VietAI/vit5-base"
tokenizer = AutoTokenizer.from_pretrained(teacher_model_name)

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"], 
        max_length=256, 
        truncation=True, 
        padding="max_length"
    )
    
    labels = tokenizer(
        text_target=examples["target_text"], 
        max_length=256, 
        truncation=True, 
        padding="max_length"
    )
    
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] 
        for label in labels["input_ids"]
    ]
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Đang mã hóa dữ liệu...")
train_dataset = train_dataset.map(preprocess_function, batched=True)
eval_dataset = eval_dataset.map(preprocess_function, batched=True)
print("Hoàn tất mã hóa!")

In [ ]:
from transformers import AutoModelForSeq2SeqLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang chạy trên thiết bị: {device}")

model = AutoModelForSeq2SeqLM.from_pretrained(teacher_model_name)

In [ ]:
import numpy as np
import json

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    def extract_elements(json_str, task_type):
        try:
            data = json.loads(json_str)
            if not isinstance(data, list): return []
            
            elements = []
            for item in data:
                if not isinstance(item, dict): continue
                
                # In thường và xóa khoảng trắng để so sánh word-by-word chuẩn tuyệt đối
                a = str(item.get("aspect_expression", "")).strip().lower()
                c = str(item.get("aspect_category", "")).strip().lower()
                s = str(item.get("sentiment", "")).strip().lower()
                
                # Bỏ qua các trường hợp NULL
                if task_type == "aspect" and a and a != "null":
                    elements.append(a)
                elif task_type == "category" and c and c != "null":
                    elements.append(c)
                elif task_type == "sentiment" and s and s != "null":
                    elements.append(s)
                elif task_type == "combined" and a and a != "null":
                    elements.append((a, c, s))
            return elements
        except Exception:
            return [] 

    counters = {
        "aspect": {"tp": 0, "fp": 0, "fn": 0},
        "category": {"tp": 0, "fp": 0, "fn": 0},
        "sentiment": {"tp": 0, "fp": 0, "fn": 0},
        "combined": {"tp": 0, "fp": 0, "fn": 0}
    }
    
    for pred_str, label_str in zip(decoded_preds, decoded_labels):
        for task in counters.keys():
            pred_set = set(extract_elements(pred_str, task))
            gold_set = set(extract_elements(label_str, task))
            
            counters[task]["tp"] += len(pred_set & gold_set) 
            counters[task]["fp"] += len(pred_set - gold_set) 
            counters[task]["fn"] += len(gold_set - pred_set) 
            
    results = {}
    for task, counts in counters.items():
        tp, fp, fn = counts["tp"], counts["fp"], counts["fn"]
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        
        results[f"{task}_f1"] = round(f1 * 100, 2) # Nhân 100 cho dễ nhìn (%)

    return results

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import shutil
import os

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/acos_model",
    evaluation_strategy="epoch",    
    learning_rate=2e-4,             
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,             
    num_train_epochs=3,             
    predict_with_generate=True, 
    generation_max_length=128,
    fp16=True,                   
    report_to="none"                
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("Bắt đầu huấn luyện mô hình ACOS...")
trainer.train()

save_dir = "/kaggle/working/acos_model_final_vit5_base"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("\nĐang nén mô hình thành file .zip...")
shutil.make_archive("/kaggle/working/acos_vit5_base_final", 'zip', save_dir)
print("Hoàn tất! Bạn có thể tải file 'acos_vit5_base_final.zip' ở cột bên phải.")